In [ ]:
import sys
sys.path.append('../src')
import utils2
import pandas as pd
import s3fs
import geopandas as gpd

arbres = pd.read_csv("https://www.data.gouv.fr/api/1/datasets/r/60433484-f30e-44ef-a362-e5553a9b7a42", sep = ";")
fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})

MY_BUCKET = "raphcrre"
PATH_IRIS = f"{MY_BUCKET}/diffusion/projet_arbres/iris.gpkg"

with fs.open(PATH_IRIS, 'rb') as f:
    iris_france = gpd.read_file(f)

iris = iris_france[iris_france['code_insee'].astype(str).str.startswith('75')].copy()
df_arbres = utils2.jointure_arbres_iris(arbres, iris)



In [ ]:
arbres


In [ ]:
# On suppose que tes colonnes s'appellent 'CIRCONFERENCE_CM' et 'HAUTEUR_M'
# On ajoute aussi le comptage pour avoir le nombre d'arbres par IRIS
iris_for_clustering = df_arbres.groupby('cleabs').agg({
    'CIRCONFERENCE (cm)': ['mean', 'std'], # Moyenne et robustesse de la taille
    'HAUTEUR (m)': ['mean', 'max'],        # Hauteur moyenne et présence de grands arbres
    'IDBASE': 'count'                    # Nombre total d'arbres
}).reset_index()

# On renomme les colonnes pour qu'elles soient plus simples à manipuler
iris_for_clustering.columns = [
    'cleabs', 'circ_moyenne', 'circ_std', 
    'haut_moyenne', 'haut_max', 'nb_arbres'
]

# Optionnel : Remplacer les NaN (IRIS avec 1 seul arbre où l'écart-type 'std' sera NaN)
iris_for_clustering = iris_for_clustering.fillna(0)


iris_for_clustering


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. Sélection des colonnes numériques uniquement
X = iris_for_clustering.drop(columns=['cleabs'])

# 2. Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Application du K-Means (testons avec 4 groupes pour commencer)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
iris_for_clustering['cluster'] = kmeans.fit_predict(X_scaled)


In [ ]:
# Voir les caractéristiques moyennes de chaque groupe
# On ne garde que les colonnes numériques pour le calcul de la moyenne
stats_clusters = iris_for_clustering.groupby('cluster').mean(numeric_only=True)

print(stats_clusters)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
# On trace la hauteur en fonction de la circonférence, colorée par cluster
sns.scatterplot(data=iris_for_clustering, 
                x='circ_moyenne', 
                y='haut_moyenne', 
                hue='cluster', 
                palette='tab10', 
                s=100, alpha=0.7)

plt.title('Profil morphologique des arbres par Cluster')
plt.xlabel('Circonférence moyenne (cm)')
plt.ylabel('Hauteur moyenne (m)')
plt.legend(title='Cluster')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


In [ ]:
# Fusion avec la géométrie des IRIS
# On utilise 'cleabs' ou 'code_iris' selon ta colonne commune
map_df = iris.merge(iris_for_clustering[['cleabs', 'cluster']], 
                    left_on='cleabs', # ou la colonne correspondante dans 'iris'
                    right_on='cleabs', 
                    how='left')

# Affichage
map_df.plot(column='cluster', cmap='tab10', legend=True, figsize=(15, 10))



In [ ]:
# On sélectionne les colonnes numériques et le cluster
cols_to_plot = ['circ_moyenne', 'haut_moyenne', 'nb_arbres', 'cluster']
sns.pairplot(iris_for_clustering[cols_to_plot], hue='cluster', palette='tab10', diag_kind='kde')
plt.show()
